# 02 Call Volume Intraday Shape Model


This notebook trains the call-volume interval branch. The model learns `slot_share`, or the share of each day's calls allocated to each half-hour slot, then applies that shape to the completed August daily anchors.

The final CV allocation is a hybrid: 40% learned ML shape and 60% historical profile. The profile gives stability for August generalization, while the ML component allows some adaptation when a day deviates from the typical weekday pattern.

## Core Implementation Used Here

The cells below call `src/pipeline.py` so the notebooks and command-line runner stay consistent. For reviewability, this section shows the exact source code for the functions used in this notebook.

```python
def build_cv_share_profiles(cv_training: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for portfolio in PORTFOLIOS:
        p = cv_training[cv_training["Portfolio"] == portfolio]
        fallback = smooth_share(p.groupby("slot")["slot_share"].median().reindex(range(SLOTS_PER_DAY), fill_value=0).to_numpy())
        for dow in range(7):
            sub = p[p["dow"] == dow]
            arr = sub.groupby("slot")["slot_share"].median().reindex(range(SLOTS_PER_DAY)).to_numpy(dtype=float)
            arr = np.where(np.isfinite(arr), arr, fallback)
            arr = smooth_share(arr)
            for slot, share in enumerate(arr):
                rows.append({"Portfolio": portfolio, "dow": dow, "slot": slot, "profile_share": share})
    return pd.DataFrame(rows)

def _daily_counts_from_share_predictions(df: pd.DataFrame, share_pred: Sequence[float]) -> np.ndarray:
    base = df.reset_index(drop=True)
    raw = np.asarray(share_pred, dtype=float)
    pred = np.zeros(len(base), dtype=float)
    for _dt, idx in base.groupby("Date").groups.items():
        idx = list(idx)
        shares = normalize(raw[idx])
        pred[idx] = shares * float(base.loc[idx[0], "daily_cv"])
    return pred

def _blend_cv_predictions(df: pd.DataFrame, ml_share_pred: Sequence[float], profile_lookup: pd.Series) -> np.ndarray:
    base = df.reset_index(drop=True)
    raw_ml = np.asarray(ml_share_pred, dtype=float)
    pred = np.zeros(len(base), dtype=float)
    for _dt, idx in base.groupby("Date").groups.items():
        idx = list(idx)
        ml_share = normalize(raw_ml[idx])
        prof_share = normalize([profile_lookup.loc[(int(base.loc[j, "dow"]), int(base.loc[j, "slot"]))] for j in idx])
        blended = normalize(CV_MODEL_WEIGHT * ml_share + CV_PROFILE_WEIGHT * prof_share)
        pred[idx] = blended * float(base.loc[idx[0], "daily_cv"])
    return pred

def train_cv_models(cv_training: pd.DataFrame) -> CVArtifacts:
    models: Dict[str, Dict[str, object]] = {}
    metrics = []
    importances = []
    profiles = build_cv_share_profiles(cv_training)

    for portfolio in PORTFOLIOS:
        p = cv_training[cv_training["Portfolio"] == portfolio].copy()
        train = p[p["Date"] < pd.Timestamp("2025-06-01")].copy()
        test = p[p["Date"] >= pd.Timestamp("2025-06-01")].copy()
        if train.empty or test.empty:
            train = p.copy()
            test = p.copy()

        X_train = train[CV_FEATURES].to_numpy(dtype=float)
        y_train = train["slot_share"].to_numpy(dtype=float)
        X_test = test[CV_FEATURES].to_numpy(dtype=float)
        y_test_counts = test["repaired_cv"].to_numpy(dtype=float)

        models_to_try = {
            "Linear Regression": LinearRegression(),
            "Decision Tree": DecisionTreeRegressor(max_depth=8, min_samples_leaf=5, random_state=42),
            "Gradient Boosting": HistGradientBoostingRegressor(
                max_iter=250,
                max_depth=4,
                learning_rate=0.05,
                min_samples_leaf=8,
                l2_regularization=1.0,
                random_state=42,
            ),
        }
        for name, model in models_to_try.items():
            model.fit(X_train, y_train)
            pred_counts = _daily_counts_from_share_predictions(test, model.predict(X_test))
            metrics.append({"Portfolio": portfolio, "model": name, "mape": mape(y_test_counts, pred_counts)})

        hgb = HistGradientBoostingRegressor(
            max_iter=250,
            max_depth=4,
            learning_rate=0.05,
            min_samples_leaf=8,
            l2_regularization=1.0,
            random_state=42,
        )
        et = ExtraTreesRegressor(n_estimators=200, max_depth=8, min_samples_leaf=5, random_state=42, n_jobs=-1)
        hgb.fit(X_train, y_train)
        et.fit(X_train, y_train)
        ensemble_share = 0.5 * np.clip(hgb.predict(X_test), 0, None) + 0.5 * np.clip(et.predict(X_test), 0, None)
        pred_counts = _daily_counts_from_share_predictions(test, ensemble_share)
        metrics.append({"Portfolio": portfolio, "model": "HGB + ExtraTrees", "mape": mape(y_test_counts, pred_counts)})

        profile_lookup = profiles[profiles["Portfolio"] == portfolio].set_index(["dow", "slot"])["profile_share"]
        blend_counts = _blend_cv_predictions(test, ensemble_share, profile_lookup)
        metrics.append({"Portfolio": portfolio, "model": "40/60 ML-profile blend", "mape": mape(y_test_counts, blend_counts)})

        for feature, importance in zip(CV_FEATURES, et.feature_importances_):
            importances.append({"Portfolio": portfolio, "feature": feature, "importance": importance})

        # Refit on all Apr-Jun repaired data for the August forecast.
        X_all = p[CV_FEATURES].to_numpy(dtype=float)
        y_all = p["slot_share"].to_numpy(dtype=float)
        hgb.fit(X_all, y_all)
        et.fit(X_all, y_all)
        models[portfolio] = {"hgb": hgb, "et": et}

    return CVArtifacts(
        models=models,
        profiles=profiles,
        metrics=pd.DataFrame(metrics),
        feature_importance=pd.DataFrame(importances),
    )

def forecast_august_cv(anchors: pd.DataFrame, artifacts: CVArtifacts) -> pd.DataFrame:
    rows = []
    profile_lookup = artifacts.profiles.set_index(["Portfolio", "dow", "slot"])["profile_share"]
    for portfolio in PORTFOLIOS:
        p_anchor = anchors[anchors["Portfolio"] == portfolio].sort_values("Date")
        for _, anchor in p_anchor.iterrows():
            dt = anchor["Date"]
            daily_cv = float(anchor["Call Volume"])
            grid = create_full_interval_grid(portfolio, dt, dt)
            grid["daily_cv"] = daily_cv
            X = grid[CV_FEATURES].to_numpy(dtype=float)
            hgb = artifacts.models[portfolio]["hgb"]
            et = artifacts.models[portfolio]["et"]
            ml_share = 0.5 * np.clip(hgb.predict(X), 0, None) + 0.5 * np.clip(et.predict(X), 0, None)
            prof_share = normalize([profile_lookup.loc[(portfolio, int(grid.loc[i, "dow"]), int(grid.loc[i, "slot"]))] for i in grid.index])
            share = normalize(CV_MODEL_WEIGHT * normalize(ml_share) + CV_PROFILE_WEIGHT * prof_share)
            cv = daily_cv * share
            for slot, value in enumerate(cv):
                rows.append(
                    {
                        "Portfolio": portfolio,
                        "Date": dt,
                        "slot": slot,
                        "interval_cv": float(value),
                        "cv_share": float(share[slot]),
                    }
                )
    return pd.DataFrame(rows)

def run_cv_pipeline() -> tuple[CVArtifacts, pd.DataFrame]:
    ensure_dirs()
    cv_training = pd.read_csv(PROCESSED_DIR / "cv_training_layer.csv", parse_dates=["Date"])
    anchors = pd.read_csv(PROCESSED_DIR / "august_daily_anchors.csv", parse_dates=["Date"])
    artifacts = train_cv_models(cv_training)
    cv_forecast = forecast_august_cv(anchors, artifacts)
    artifacts.profiles.to_csv(PROCESSED_DIR / "cv_share_profiles.csv", index=False)
    artifacts.metrics.to_csv(OUTPUT_DIR / "cv_holdout_metrics.csv", index=False)
    artifacts.feature_importance.to_csv(OUTPUT_DIR / "cv_feature_importance.csv", index=False)
    cv_forecast.to_csv(OUTPUT_DIR / "cv_interval_forecast_unbiased.csv", index=False)
    return artifacts, cv_forecast
```


In [ ]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'src' / 'pipeline.py').exists():
        ROOT = candidate
        break
    if (candidate / 'datathon_final' / 'src' / 'pipeline.py').exists():
        ROOT = candidate / 'datathon_final'
        break
if ROOT is None:
    raise RuntimeError('Could not find datathon_final project root.')
sys.path.insert(0, str(ROOT))
from src import pipeline


## Train and forecast

The model is portfolio-specific. It uses interval-positioning features (`slot`, `is_peak`, cyclic slot/weekday encodings) plus `daily_cv` as a secondary conditioning feature for day size.

In [ ]:
artifacts, cv_forecast = pipeline.run_cv_pipeline()
print(f'CV forecast rows: {len(cv_forecast):,}')
cv_forecast.head()

## Holdout comparison

The comparison below uses an Apr-May training window and a June holdout. The final forecast uses the `40/60 ML-profile blend` because it balances historical profile stability with learned adaptability.

In [ ]:
metrics = artifacts.metrics.sort_values(['Portfolio', 'mape'])
metrics

## Feature importance

The feature importances come from the ExtraTrees component of the CV share model. Time-of-day structure dominates the shape model, which matches the business framing: the CV branch is primarily learning where each half-hour sits within the daily/weekly demand cycle.

In [ ]:
(artifacts.feature_importance
 .groupby('feature', as_index=False)['importance'].mean()
 .sort_values('importance', ascending=False))